# UdaPlay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside the `games` folder. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [ ]:
# Optional SQLite compatibility for the Udacity workspace.
# pysqlite3 is imported dynamically so local editors do not warn when it is absent.

from importlib import import_module, util
import sys

if util.find_spec("pysqlite3") is not None:
    pysqlite3 = import_module("pysqlite3")
    sys.modules["sqlite3"] = pysqlite3

In [1]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

Create a local `.env` file using `.env.example` as a guide. The real `.env` file is intentionally ignored by Git.

In [2]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "https://openai.vocareum.com/v1")

assert OPENAI_API_KEY is not None, "OPENAI_API_KEY must be set in your local .env file"
assert TAVILY_API_KEY is not None, "TAVILY_API_KEY must be set in your local .env file"

### VectorDB Instance

In [3]:
chroma_client = chromadb.PersistentClient(path="chromadb")

### Collection

In [4]:
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY,
    api_base=OPENAI_BASE_URL,
)

In [5]:
collection = chroma_client.get_or_create_collection(
    name="udaplay",
    embedding_function=embedding_fn,
)

### Add documents

In [6]:
# Make sure you have a directory "games"
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # You can change what text you want to index
    content = (
        f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - "
        f"{game['Genre']}, published by {game['Publisher']}. {game['Description']}"
    )

    # Use file name (like 001) as ID
    doc_id = os.path.splitext(file_name)[0]

    # upsert overwrites existing IDs, so re-running re-indexes with the latest content
    collection.upsert(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )


### Semantic Search

Confirm the vector database can be queried by semantic similarity.


In [7]:
# Query the collection with a natural language question (not a keyword match)
results = collection.query(
    query_texts=["A racing game for the PlayStation"],
    n_results=3,
)

for doc, distance in zip(results["documents"][0], results["distances"][0]):
    print(f"distance={distance:.4f}  {doc}")


distance=0.1084  [PlayStation 3] Gran Turismo 5 (2010) - Racing, published by Sony Computer Entertainment. A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving physics.
distance=0.1098  [PlayStation 1] Gran Turismo (1997) - Racing, published by Sony Computer Entertainment. A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.
distance=0.1709  [Nintendo Switch] Mario Kart 8 Deluxe (2017) - Racing, published by Nintendo. An enhanced version of Mario Kart 8, featuring new characters, tracks, and improved gameplay mechanics.


### Richer Queries on the Personalized Dataset

The dataset was extended with ~145 PC games (indie, horror, cozy, narrative, etc.).
Because retrieval is semantic, we can search by *mood* or *theme* rather than exact titles.

In [8]:
# Each query is a theme/mood, not a title -- semantic search finds the closest games
queries = [
    "a cozy game about making coffee and listening to people's stories",
    "a scary co-op horror game to play with friends",
    "a relaxing game about building little towns",
]

for q in queries:
    print(f"\nQuery: {q!r}")
    results = collection.query(query_texts=[q], n_results=3)
    for meta, distance in zip(results["metadatas"][0], results["distances"][0]):
        print(f"  distance={distance:.4f}  {meta['Name']} ({meta['Genre']})")


Query: "a cozy game about making coffee and listening to people's stories"
  distance=0.1302  Coffee Talk (Visual novel)
  distance=0.1403  Coffee Talk Tokyo (Visual novel)
  distance=0.1549  Coffee Talk Episode 2: Hibiscus & Butterfly (Visual novel)

Query: 'a scary co-op horror game to play with friends'
  distance=0.1306  Phasmophobia (Co-op horror)
  distance=0.1318  R.E.P.O. (Co-op horror)
  distance=0.1328  Lethal Company (Co-op horror)

Query: 'a relaxing game about building little towns'
  distance=0.1209  Tile Cities (Simulation)
  distance=0.1217  Townscaper (Sandbox)
  distance=0.1479  SUMMERHOUSE (Sandbox)
